# Detective R-GNN — Stage 1 Evaluation (pre-PPO)

Carica il best checkpoint BC e fa giocare la GNN (argmax sulla policy, no sampling) contro MCTS Mr.X. Confronto **paired** contro la heuristic baseline (`Game.detective_automated_turn`): stesso seed di game → stessa posizione iniziale → confronto detective-strategy a parità di setup.

**Cosa misura**:
- Winrate detective GNN vs winrate detective euristica
- Lunghezza media partita
- Distribuzione vittorie per turno di cattura
- Diagnostica su partite identiche (entrambi vinti / entrambi persi / solo GNN / solo euristica)

**Due livelli di Mr.X**:
1. Weakened (explo=8, sims=8) — stesso del dataset di training → check di sanity
2. Full strength (explo=15, sims=25) — il vero opponente che dovremo battere in PPO

Se la GNN è già vicina o sopra l'euristica nel livello 1, OK per partire con PPO. Se perde male in entrambi, c'è un bug di inference da indagare prima.

## 1. Setup

In [ ]:
%%bash
set -e
cd /content
if [ ! -d scotland_yard ]; then
  git clone --depth 1 https://github.com/Jacopo888/scotland_yard.git
fi
ls scotland_yard

In [ ]:
!pip install -q networkx numpy pandas tqdm scipy

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, glob
DRIVE_BASE = '/content/drive/MyDrive/scotland_yard_dataset'
CHECKPOINT_DIR = os.path.join(DRIVE_BASE, 'checkpoints')

ckpts = sorted(glob.glob(os.path.join(CHECKPOINT_DIR, 'rgnn_bc_best_*.pt')))
assert ckpts, f'Nessun checkpoint BC in {CHECKPOINT_DIR} — fai prima il training'
CKPT_PATH = ckpts[-1]
print('Latest BC checkpoint:', CKPT_PATH)

## 2. Imports + repo path

In [ ]:
import os, sys, json, time, random
from datetime import datetime
from collections import Counter

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

REPO_DIR = '/content/scotland_yard'
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path: sys.path.insert(0, REPO_DIR)

from game import Game, ADJ, BOARD_GRAPH, SHORTEST_PATH_TENSOR, ticket_bitmask
from detective_engine import DetectiveEngine
import mrx_engine
from mrx_engine import MrxEngine

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

## 3. Static graph + reveal schedule

Replicato esattamente dal training notebook (cell 8) per garantire che il modello veda gli stessi input.

In [ ]:
N_NODES = 199
RELATIONS = ('taxi', 'bus', 'underground', 'water')
REL2IDX = {r: i for i, r in enumerate(RELATIONS)}
VEHICLES = ('taxi', 'bus', 'underground')
VEH2IDX = {v: i for i, v in enumerate(VEHICLES)}
TICKET_VOCAB = {'': 0, 'taxi': 1, 'bus': 2, 'underground': 3, 'water': 4}
MAX_TURN = 22
REVEAL_TURNS = (3, 8, 13, 18)

def turns_to_next_reveal(t):
    for r in REVEAL_TURNS:
        if r > t: return r - t
    return -1


def build_dense_adj():
    A = np.zeros((len(RELATIONS), N_NODES, N_NODES), dtype=np.float32)
    for v_str, neigh in ADJ.items():
        v = int(v_str) - 1
        for u_str, types in neigh.items():
            u = int(u_str) - 1
            for t in types:
                if t in REL2IDX:
                    A[REL2IDX[t], v, u] = 1.0
    deg = A.sum(axis=2, keepdims=True); deg[deg == 0] = 1.0
    return A / deg

def build_node_static_features():
    deg = np.zeros((N_NODES, len(RELATIONS)), dtype=np.float32)
    for v_str, neigh in ADJ.items():
        v = int(v_str) - 1
        for u_str, types in neigh.items():
            for t in types:
                if t in REL2IDX:
                    deg[v, REL2IDX[t]] += 1.0
    has_und = (deg[:, REL2IDX['underground']] > 0).astype(np.float32)[:, None]
    return np.concatenate([deg, has_und], axis=1)

def build_edge_lookup():
    lookup = {}
    for v_str, neigh in ADJ.items():
        v = int(v_str) - 1
        out = []
        for u_str, types in neigh.items():
            u = int(u_str) - 1
            rel_idx = sorted({REL2IDX[t] for t in types if t in REL2IDX})
            out.append((u, rel_idx))
        lookup[v] = out
    return lookup

DENSE_ADJ = torch.from_numpy(build_dense_adj()).to(DEVICE)
NODE_STATIC = torch.from_numpy(build_node_static_features()).to(DEVICE)
EDGE_LOOKUP = build_edge_lookup()
_SP_ANY_NP = SHORTEST_PATH_TENSOR[7, :199, :199].astype(np.float32)

NODE_DYN_DIM = 1 + 5 + 1 + 1 + 5 + 1   # 14
GLOBAL_FEATURE_DIM = 1 + 1 + 4 + 15 + 5  # 26
print('DENSE_ADJ', DENSE_ADJ.shape, 'NODE_STATIC', NODE_STATIC.shape, 'SP_ANY_NP', _SP_ANY_NP.shape)

## 4. Model architecture (DenseRGCN + DetectiveRGNN)

Stesso codice del training notebook (cell 18).

In [ ]:
class DenseRGCNLayer(nn.Module):
    def __init__(self, in_dim, out_dim, n_relations, dropout=0.1):
        super().__init__()
        self.W0 = nn.Linear(in_dim, out_dim, bias=True)
        self.Wr = nn.Parameter(torch.empty(n_relations, in_dim, out_dim))
        nn.init.xavier_uniform_(self.Wr)
        self.dropout = nn.Dropout(dropout)
        self.residual = (in_dim == out_dim)
    def forward(self, h, A):
        self_msg = self.W0(h)
        AH = torch.einsum('rmn,bnk->rbmk', A, h)
        AHW = torch.einsum('rbmk,rkj->rbmj', AH, self.Wr)
        rel_msg = AHW.sum(dim=0)
        out = F.relu(self_msg + rel_msg)
        out = self.dropout(out)
        if self.residual: out = out + h
        return out


class DetectiveRGNN(nn.Module):
    def __init__(self, node_dyn_dim, node_static_dim, global_dim,
                 hidden=64, n_layers=3, n_relations=4, det_emb_dim=8,
                 edge_emb_dim=8, dropout=0.1):
        super().__init__()
        self.input_proj = nn.Linear(node_dyn_dim + node_static_dim, hidden)
        self.layers = nn.ModuleList([
            DenseRGCNLayer(hidden, hidden, n_relations, dropout=dropout)
            for _ in range(n_layers)
        ])
        self.det_id_emb = nn.Embedding(5, det_emb_dim)
        self.edge_type_emb = nn.Embedding(n_relations, edge_emb_dim)

        ph_in = 2*hidden + edge_emb_dim + global_dim + det_emb_dim
        self.policy_mlp = nn.Sequential(
            nn.Linear(ph_in, hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, 1),
        )

        vh_in = 2*hidden + global_dim + det_emb_dim
        self.value_mlp = nn.Sequential(
            nn.Linear(vh_in, hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, 1),
        )

    def encode(self, node_dyn, node_static_b, A):
        x = torch.cat([node_dyn, node_static_b], dim=-1)
        h = self.input_proj(x)
        for layer in self.layers:
            h = layer(h, A)
        return h

    def forward(self, batch, A, node_static):
        B = batch['node_dyn'].shape[0]
        node_dyn = batch['node_dyn']
        glob = batch['glob']
        det_id = batch['det_id']
        ego_pos = batch['ego_pos']
        legal_neigh = batch['legal_neighbors_mask']

        node_static_b = node_static.unsqueeze(0).expand(B, -1, -1)
        h = self.encode(node_dyn, node_static_b, A)
        det_e = self.det_id_emb(det_id)
        glob_full = torch.cat([glob, det_e], dim=-1)

        h_mean = h.mean(dim=1)
        h_max = h.max(dim=1).values
        v_in = torch.cat([h_mean, h_max, glob_full], dim=-1)
        value = self.value_mlp(v_in).squeeze(-1)

        all_logits = []
        all_cand = []
        for b in range(B):
            v = int(ego_pos[b].item())
            mask = legal_neigh[b].cpu().numpy() if legal_neigh.is_cuda else legal_neigh[b].numpy()
            cand = [(u, rels) for (u, rels) in EDGE_LOOKUP[v]
                    if u < N_NODES and mask[u]]
            if not cand:
                all_logits.append(torch.zeros(1, device=h.device))
                all_cand.append([])
                continue
            us = torch.tensor([u for (u, _) in cand], device=h.device, dtype=torch.long)
            edge_e = torch.stack([self.edge_type_emb(
                torch.tensor(rels, device=h.device, dtype=torch.long)).sum(dim=0)
                for (_, rels) in cand], dim=0)
            h_v = h[b, v].unsqueeze(0).expand(len(cand), -1)
            h_u = h[b, us]
            g_b = glob_full[b].unsqueeze(0).expand(len(cand), -1)
            feat = torch.cat([h_v, h_u, edge_e, g_b], dim=-1)
            logits = self.policy_mlp(feat).squeeze(-1)
            all_logits.append(logits)
            all_cand.append([u for (u, _) in cand])
        return value, all_logits, all_cand

## 5. Carica il checkpoint BC

In [ ]:
# weights_only=False: il checkpoint contiene np.float64 (numpy scalars).
# PyTorch >= 2.6 di default rifiuta unpickling di numpy types; e' un checkpoint nostro, lo possiamo fidare.
ckpt = torch.load(CKPT_PATH, map_location=DEVICE, weights_only=False)
cfg = ckpt.get('config', {})
print('=== Checkpoint metadata ===')
for k in ('epoch', 'val_nll', 'val_acc', 'val_mse', 'rtg_mean', 'rtg_std'):
    if k in ckpt: print(f'  {k}: {ckpt[k]}')
print('  config:', cfg)

model = DetectiveRGNN(
    node_dyn_dim=cfg.get('node_dyn_dim', NODE_DYN_DIM),
    node_static_dim=cfg.get('node_static_dim', int(NODE_STATIC.shape[1])),
    global_dim=cfg.get('global_dim', GLOBAL_FEATURE_DIM),
    hidden=cfg.get('hidden', 64),
    n_layers=cfg.get('n_layers', 3),
    n_relations=cfg.get('n_relations', 4),
    dropout=0.0,   # eval mode
).to(DEVICE)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
n_params = sum(p.numel() for p in model.parameters())
print(f'Model loaded: {n_params/1e3:.1f}k params, set to eval()')

## 6. Inference helper — `policy_act` (argmax, NO leakage)

`revealed_positions` deve essere passato esplicitamente (lista delle posizioni Mr.X rivelate ai turni 3/8/13/18); il caller è responsabile di tracciarle. `last_mrx_ticket` osservabile.

In [ ]:
def collate_one(sample):
    return {
        'node_dyn': sample['node_dyn'].unsqueeze(0),
        'glob': sample['glob'].unsqueeze(0),
        'det_id': torch.tensor([sample['det_id']], dtype=torch.long),
        'ego_pos': torch.tensor([sample['ego_pos']], dtype=torch.long),
        'legal_neighbors_mask': sample['legal_neighbors_mask'].unsqueeze(0),
    }

@torch.no_grad()
def policy_act(model, game, engine, det_id, revealed_positions=(), last_mrx_ticket=''):
    """GNN inference per un detective. Argmax sulla policy.
    Ritorna la destinazione come stringa, per restare compatibile con Game/ADJ.

    revealed_positions: iterable di posizioni Mr.X (1-indexed) rivelate ai turni 3/8/13/18.
    """
    mrx_visited_mask = np.zeros(N_NODES, dtype=np.float32)
    for p in revealed_positions:
        mrx_visited_mask[int(p) - 1] = 1.0

    det_pos_arr = np.array([int(p) - 1 for p in game.detectives_pos], dtype=np.int64)
    ego = det_pos_arr[det_id]
    belief = engine.belief_state.astype(np.float32)
    is_det = np.zeros((5, N_NODES), dtype=np.float32)
    for k, p in enumerate(det_pos_arr): is_det[k, p] = 1.0
    is_ego = np.zeros(N_NODES, dtype=np.float32); is_ego[ego] = 1.0
    dpd = np.stack([_SP_ANY_NP[p] for p in det_pos_arr], axis=1).astype(np.float32)
    md = dpd.min(axis=1, keepdims=True)
    node_dyn = np.concatenate([
        belief[:, None], is_det.T, is_ego[:, None], md, dpd, mrx_visited_mask[:, None]
    ], axis=1).astype(np.float32)

    # legal mask
    legal = np.zeros((N_NODES, 3), dtype=bool)
    occupied = set(game.detectives_pos[:det_id] + game.detectives_pos[det_id + 1:])
    tickets = game.detective_tickets[det_id]
    for nb_, types in ADJ[game.detectives_pos[det_id]].items():
        if nb_ in occupied: continue
        for t in types:
            if t in VEH2IDX and tickets.get(t, 0) > 0:
                legal[int(nb_) - 1, VEH2IDX[t]] = True
    legal_neigh = legal.any(axis=1)
    cand = [u for (u, _) in EDGE_LOOKUP[ego] if legal_neigh[u]]
    if not cand:
        return None, None

    last_oh = np.zeros(5, dtype=np.float32); last_oh[TICKET_VOCAB.get(last_mrx_ticket, 0)] = 1.0
    all_dt = np.array([[t['taxi'], t['bus'], t['underground']] for t in game.detective_tickets],
                      dtype=np.float32) / 10.0
    glob = np.concatenate([
        np.array([game.turn / MAX_TURN, (turns_to_next_reveal(game.turn) / 5.0)], dtype=np.float32),
        np.array([game.mrx_tickets[v] for v in ('taxi','bus','underground','water')],
                 dtype=np.float32) / 10.0,
        all_dt.reshape(-1),
        last_oh,
    ])

    batch = collate_one({
        'node_dyn': torch.from_numpy(node_dyn),
        'glob': torch.from_numpy(glob),
        'det_id': det_id,
        'ego_pos': int(ego),
        'legal_neighbors_mask': torch.from_numpy(legal_neigh),
    })
    for k, v in batch.items():
        if torch.is_tensor(v): batch[k] = v.to(DEVICE)
    _, all_logits, all_cand = model(batch, DENSE_ADJ, NODE_STATIC)
    logits = all_logits[0]
    cand_model = all_cand[0]
    # cand del modello == cand calcolato qui (stessa legal_neigh + EDGE_LOOKUP)
    best_idx = int(torch.argmax(logits).item())
    best_dest_int = int(cand_model[best_idx]) + 1
    chosen_v = next(v for v in ('taxi','bus','underground') if legal[best_dest_int - 1, VEH2IDX[v]])
    return str(best_dest_int), chosen_v

# Smoke test
_g = Game(); _e = DetectiveEngine(_g.detectives_pos)
print('policy_act smoke test:', policy_act(model, _g, _e, det_id=0, revealed_positions=()))

## 7. Play loops

Due funzioni che replicano `main.py::play()` ma con detective strategy sostituibile.

In [ ]:
def play_heuristic(mrx_explo=15, mrx_sims=25):
    """Detectives = Game.detective_automated_turn (argmax on belief + shortest path).
    Returns dict with winner, final_turn, mrx_pos, detectives_pos."""
    mrx_engine.NUM_EXPLORATIONS = mrx_explo
    mrx_engine.NUM_SIMULATIONS = mrx_sims

    game = Game()
    engine = DetectiveEngine(game.detectives_pos)
    mrx = MrxEngine(game, engine)

    while True:
        if game.check_victory(silent=True):
            break

        game_over = False
        for i in range(game.num_detectives):
            game.detective_automated_turn(engine.belief_state, i)
            if game.check_victory(silent=True):
                game_over = True; break
            engine.kalman_filter()
        game.detectives_moves.append(game.detectives_pos[:])
        if game_over: break

        best_pos, best_ticket = mrx.search()
        game.x_automated_turn(best_pos, best_ticket)
        if game.check_victory(silent=True): break

        engine.update_belief_after_mrx_move(best_ticket)
        if (game.turn - 3) % 5 == 0:
            engine.mrx_is_spotted(game.mrx_pos)

    return {
        'winner': game.winner,
        'final_turn': game.turn,
        'mrx_pos': game.mrx_pos,
        'detectives_pos': list(game.detectives_pos),
    }


def play_gnn(model, mrx_explo=15, mrx_sims=25):
    """Detectives = GNN argmax (no sampling). Returns same dict as play_heuristic.
    revealed_positions e last_mrx_ticket sono tracciati esplicitamente per evitare leakage."""
    mrx_engine.NUM_EXPLORATIONS = mrx_explo
    mrx_engine.NUM_SIMULATIONS = mrx_sims

    game = Game()
    engine = DetectiveEngine(game.detectives_pos)
    mrx = MrxEngine(game, engine)

    revealed_positions = []
    last_mrx_ticket = ''

    while True:
        if game.check_victory(silent=True):
            break

        game_over = False
        for det_id in range(game.num_detectives):
            best_dest, vehicle = policy_act(
                model, game, engine, det_id,
                revealed_positions=tuple(revealed_positions),
                last_mrx_ticket=last_mrx_ticket,
            )
            if best_dest is None:
                # detective bloccato: niente mossa
                if game.check_victory(silent=True):
                    game_over = True; break
                continue
            pos = game.detectives_pos[det_id]
            game.use_ticket(det_id, pos, best_dest)
            game.detectives_pos[det_id] = best_dest
            if game.check_victory(silent=True):
                game_over = True; break
            engine.kalman_filter()
        game.detectives_moves.append(game.detectives_pos[:])
        if game_over: break

        best_pos, best_ticket = mrx.search()
        game.x_automated_turn(best_pos, best_ticket)
        if best_ticket is not None:
            last_mrx_ticket = best_ticket
        if game.check_victory(silent=True): break

        engine.update_belief_after_mrx_move(best_ticket)
        if (game.turn - 3) % 5 == 0:
            engine.mrx_is_spotted(game.mrx_pos)
            revealed_positions.append(int(game.mrx_pos))

    return {
        'winner': game.winner,
        'final_turn': game.turn,
        'mrx_pos': game.mrx_pos,
        'detectives_pos': list(game.detectives_pos),
    }

# Smoke test 1 partita per strategia
print('Smoke test (1 game per strategy, weakened Mr.X):')
random.seed(0); np.random.seed(0)
_h = play_heuristic(mrx_explo=8, mrx_sims=8)
print(f'  heuristic: winner={_h["winner"]}  final_turn={_h["final_turn"]}')
random.seed(0); np.random.seed(0)
_g = play_gnn(model, mrx_explo=8, mrx_sims=8)
print(f'  GNN:       winner={_g["winner"]}  final_turn={_g["final_turn"]}')

## 8. Evaluation paired: stessa seed → stessa posizione iniziale

Per ogni seed in `[0..N-1]`, gioca prima euristica poi GNN ri-seedando `random` / `np.random`. La `Game()` init usa `random.choice` + `random.sample` quindi le posizioni iniziali coincidono.

In [ ]:
N_GAMES = 150       # 150 paired = 300 partite totali per livello
MRX_CONFIGS = [
    ('weakened (training dist)', 8, 8),
    ('full strength',            15, 25),
]

def wilson_ci(wins, n, alpha=0.05):
    """Wilson score interval per binomial proportion."""
    from scipy.stats import norm
    if n == 0: return (0.0, 1.0)
    z = norm.ppf(1 - alpha/2)
    p = wins / n
    denom = 1 + z**2/n
    center = (p + z**2/(2*n)) / denom
    spread = z * np.sqrt(p*(1-p)/n + z**2/(4*n**2)) / denom
    return max(0.0, center - spread), min(1.0, center + spread)

all_results = {}
for label, explo, sims in MRX_CONFIGS:
    print(f'\n{"="*60}')
    print(f'Mr.X: {label}  (NUM_EXPLORATIONS={explo}, NUM_SIMULATIONS={sims})')
    print(f'{"="*60}')

    h_results, g_results = [], []
    for seed in tqdm(range(N_GAMES), desc=f'{label}'):
        random.seed(seed); np.random.seed(seed)
        h_results.append(play_heuristic(mrx_explo=explo, mrx_sims=sims))
        random.seed(seed); np.random.seed(seed)
        g_results.append(play_gnn(model, mrx_explo=explo, mrx_sims=sims))

    h_wins = sum(r['winner'] == 0 for r in h_results)
    g_wins = sum(r['winner'] == 0 for r in g_results)
    h_lo, h_hi = wilson_ci(h_wins, N_GAMES)
    g_lo, g_hi = wilson_ci(g_wins, N_GAMES)

    # Tabella paired
    both_won = sum(1 for h, g in zip(h_results, g_results) if h['winner'] == 0 and g['winner'] == 0)
    only_h_won = sum(1 for h, g in zip(h_results, g_results) if h['winner'] == 0 and g['winner'] == 1)
    only_g_won = sum(1 for h, g in zip(h_results, g_results) if h['winner'] == 1 and g['winner'] == 0)
    both_lost = sum(1 for h, g in zip(h_results, g_results) if h['winner'] == 1 and g['winner'] == 1)

    print(f'\nDetective winrate (95% CI Wilson):')
    print(f'  Heuristic: {h_wins:3d}/{N_GAMES}  =  {h_wins/N_GAMES*100:5.1f}%   [{h_lo*100:4.1f}, {h_hi*100:4.1f}]')
    print(f'  GNN:       {g_wins:3d}/{N_GAMES}  =  {g_wins/N_GAMES*100:5.1f}%   [{g_lo*100:4.1f}, {g_hi*100:4.1f}]')
    delta_pp = (g_wins - h_wins) / N_GAMES * 100
    print(f'  Delta:     {delta_pp:+.1f} pp  (GNN - heuristic)')

    print(f'\nPaired contingency (stessa seed):')
    print(f'  both win :  {both_won:3d}   (entrambi catturano Mr.X)')
    print(f'  only GNN :  {only_g_won:3d}   (GNN cattura, euristica no)')
    print(f'  only heu :  {only_h_won:3d}   (euristica cattura, GNN no)')
    print(f'  both lose:  {both_lost:3d}   (entrambi falliscono)')

    # McNemar test (one-sided): la differenza only_g_won vs only_h_won e' significativa?
    from scipy.stats import binomtest
    disc = only_g_won + only_h_won
    if disc > 0:
        pval = binomtest(only_g_won, disc, p=0.5, alternative='two-sided').pvalue
        print(f'  McNemar p-value (two-sided): {pval:.4f}')

    print(f'\nLunghezza media partita:')
    print(f'  Heuristic: mean={np.mean([r["final_turn"] for r in h_results]):.1f}  '
          f'std={np.std([r["final_turn"] for r in h_results]):.1f}')
    print(f'  GNN:       mean={np.mean([r["final_turn"] for r in g_results]):.1f}  '
          f'std={np.std([r["final_turn"] for r in g_results]):.1f}')

    all_results[label] = {
        'h_results': h_results, 'g_results': g_results,
        'h_wins': h_wins, 'g_wins': g_wins, 'n': N_GAMES,
        'both_won': both_won, 'only_g_won': only_g_won,
        'only_h_won': only_h_won, 'both_lost': both_lost,
    }

## 9. Verdetto + salvataggio risultati

In [ ]:
print('=' * 60)
print('VERDETTO Stage 1 BC')
print('=' * 60)
for label, r in all_results.items():
    delta = (r['g_wins'] - r['h_wins']) / r['n'] * 100
    if delta >= -2:
        status = 'OK'
        msg = 'GNN >= euristica - 2pp. Pronto per PPO.'
    elif delta >= -10:
        status = 'BORDERLINE'
        msg = "GNN perde rispetto alla euristica ma di poco. Puoi partire con PPO ma aspettati piu iterazioni per recuperare."
    else:
        status = 'FAIL'
        msg = 'GNN perde nettamente. Sospetto bug di inference (input mismatch dataset/runtime, dropout, leakage residuo). Investigare prima di PPO.'
    print(f'\n{label}: delta={delta:+.1f}pp  -->  [{status}]  {msg}')

# Salva risultati su Drive
OUT_DIR = os.path.join(DRIVE_BASE, 'eval_results')
os.makedirs(OUT_DIR, exist_ok=True)
RUN_TAG = datetime.now().strftime('%Y%m%d_%H%M%S')
summary = {
    'checkpoint': CKPT_PATH,
    'checkpoint_meta': {k: float(ckpt[k]) if isinstance(ckpt.get(k), (int, float)) else ckpt.get(k)
                        for k in ('epoch', 'val_nll', 'val_acc', 'val_mse', 'rtg_mean', 'rtg_std')
                        if k in ckpt},
    'n_games': N_GAMES,
    'configs': {label: {k: v for k, v in r.items() if k not in ('h_results', 'g_results')}
                for label, r in all_results.items()},
    'generated_at': datetime.now().isoformat(),
}
out_path = os.path.join(OUT_DIR, f'eval_{RUN_TAG}.json')
with open(out_path, 'w') as f:
    json.dump(summary, f, indent=2, default=str)
print(f'\nSummary salvato: {out_path}')

## 10. (Opzionale) Diagnostica visiva

Distribuzione turno di cattura per i due strategy. Detective che vincono prima → meglio.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, len(MRX_CONFIGS), figsize=(6 * len(MRX_CONFIGS), 4), sharey=True)
if len(MRX_CONFIGS) == 1: axes = [axes]

for ax, (label, r) in zip(axes, all_results.items()):
    h_turns_win = [x['final_turn'] for x in r['h_results'] if x['winner'] == 0]
    g_turns_win = [x['final_turn'] for x in r['g_results'] if x['winner'] == 0]
    bins = np.arange(0, 24, 2)
    ax.hist([h_turns_win, g_turns_win], bins=bins, label=['heuristic', 'GNN'], alpha=0.7)
    ax.set_title(f'Mr.X: {label}\nturno di cattura (solo wins)')
    ax.set_xlabel('turno'); ax.set_ylabel('# partite')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout(); plt.show()